# Ames Housing — Pipeline, ColumnTransformer, and Ridge vs. LinearRegression

Applying Pipeline, SimpleImputer, and ColumnTransformer to Kaggle's Ames Housing dataset
(1460 houses, 81 columns), then comparing Ridge Regression against a plain LinearRegression
baseline under 5-fold cross-validated RMSE.

Requires `ames_train.csv` in `../data/` relative to this notebook.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

### Loading Data

In [2]:
df = pd.read_csv('../data/ames_train.csv')
print(f"Shape: {df.shape}")
df.head(3)

Shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500


### The Nan-doesn't-mean-missing trap

Several columns use NaN to mean "this feature doesn't exist for this house" (no pool, no
alley, no fence, no fireplace, no garage), not "value unknown." Confirming this before
deciding what to impute avoids inventing values for houses that simply lack a given feature.

In [3]:
df.isna().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageYrBlt       81
GarageCond        81
GarageType        81
GarageFinish      81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Id                 0
dtype: int64

In [4]:
na_means_absent = ['PoolQC', 'Alley', 'Fence', 'FireplaceQu', 'GarageType']
for col in na_means_absent:
    print(f"{col}: {df[col].isna().sum()} NaN out of {len(df)}  (NaN = feature absent, not unknown)")

print()
genuinely_missing = ['LotFrontage', 'Electrical']
for col in genuinely_missing:
    print(f"{col}: {df[col].isna().sum()} NaN out of {len(df)}  (NaN = genuinely unrecorded)")

PoolQC: 1453 NaN out of 1460  (NaN = feature absent, not unknown)
Alley: 1369 NaN out of 1460  (NaN = feature absent, not unknown)
Fence: 1179 NaN out of 1460  (NaN = feature absent, not unknown)
FireplaceQu: 690 NaN out of 1460  (NaN = feature absent, not unknown)
GarageType: 81 NaN out of 1460  (NaN = feature absent, not unknown)

LotFrontage: 259 NaN out of 1460  (NaN = genuinely unrecorded)
Electrical: 1 NaN out of 1460  (NaN = genuinely unrecorded)


### Selecting a Feature Subset

81 columns is too many to hand-reason about, picking a mix of intuitive predictors (size,
quality, location) rather than throwing every column in.

In [5]:
numeric_features = [
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt',
    'TotalBsmtSF', 'GrLivArea', 'FullBath', 'BedroomAbvGr',
    'GarageCars', 'GarageArea'
]

categorical_features = [
    'MSZoning', 'Neighborhood', 'HouseStyle', 'ExterQual',
    'KitchenQual', 'SaleCondition'
]

target = 'SalePrice'

X = df[numeric_features + categorical_features]
y = df[target]

In [6]:
print(f"Missing values in selected numeric columns:\n{X[numeric_features].isna().sum()[X[numeric_features].isna().sum() > 0]}")
print(f"\nMissing values in selected categorical columns:\n{X[categorical_features].isna().sum()[X[categorical_features].isna().sum() > 0]}")

Missing values in selected numeric columns:
Series([], dtype: int64)

Missing values in selected categorical columns:
Series([], dtype: int64)


### Train/Test Split

Splitting before any preprocessing is fit, so the imputer, scaler, and encoder in the
upcoming ColumnTransformer only ever see training data until prediction time.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (1168, 16) | Test: (292, 16)


### Building the ColumnTransformer

Each branch (numeric, categorical) is itself a small Pipeline: impute, then scale/encode.
Both branches run in parallel and get concatenated back into one array.

In [8]:
numeric_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

categorical_transformer = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [9]:
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)
print(f"Original feature count: {len(numeric_features + categorical_features)}")
print(f"Transformed shape: {X_train_transformed.shape}")

Original feature count: 16
Transformed shape: (1168, 62)
